# M1. Network performance patterns


Large event-based networks are fastest when they expose large tensor operations rather than many tiny Python objects.

## Minimize mechanism-object count

Use banked point-process mechanisms when many colocated synapses share the same mechanism class:

```python
post.dend.insert(exp2syn, alias="exc", copies=1000)
slots = post.dend.slots(post.mech.exc)
```

The local slots remain independent, but one vectorized mechanism object updates them.

## Minimize NetCon group count

Group homogeneous source/target/synapse-class blocks into one NetCon where possible. Keep separate NetCons when the source population, target mechanism, presynaptic variable, thresholding mode, delay backend, or differentiability requirements differ.

## Use shared event variables

When many outgoing connections share a voltage threshold, insert one detector and connect from the detector's event variable:

```python
pre.soma.insert(spikedetect, alias="spikedetect", threshold=-30.0)
net.connect_one_to_one_slots(..., pre_var="mech.spikedetect.spikes", threshold=None)
```

This avoids repeated threshold-state allocation in each NetCon.

## Choose the delay backend deliberately

- `dense`: regular and differentiable; often best for training and GPU workloads when memory is available.
- `sparse_calendar`: lower memory for sparse CPU event traffic.
- `bitpacked_history`: compact source-spike history for eligible SNN-style inference.

Inspect the built NetCons before optimizing; actual memory use depends on connection count, target slot count, delay range, and event tracking.

## Batch independent experiments

Prefer `Network.batch(n)` for independent replicas. It preserves the network structure while adding a leading batch dimension, so the same compiled/optimized operations can be reused across conditions.